# 03 · Can a Neural Network Predict Tomorrow's Regime?
**Week 3 of _Can a Neural Network See Market Regimes Coming?_**

The HMM from notebook 02 tells us which regime each day was in. Here we treat the HMM as a **teacher**:
a neural network looks at the **last 20 days** of returns and volatility and learns to predict **tomorrow's regime**.

This "teacher → student" setup is a common trick in machine learning (it is also how some robot-swarm
policies are trained: a network imitates an expert that has more information).

**The honest question:** does the network do better than the simplest possible guess,
*"tomorrow's regime = today's regime"*? Regimes are sticky, so that guess is surprisingly hard to beat.

In [ ]:
!pip -q install yfinance hmmlearn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from hmmlearn.hmm import GaussianHMM
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

np.random.seed(0)
torch.manual_seed(0)
plt.rcParams["figure.figsize"] = (11, 4)

## 1. Data, features and HMM labels (from notebooks 01–02)

In [ ]:
def load_close(ticker, start="2008-01-01"):
    """Download daily closing prices from Yahoo Finance as a pandas Series."""
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    close = df["Close"]
    if isinstance(close, pd.DataFrame):   # newer yfinance returns a 1-column DataFrame
        close = close.iloc[:, 0]
    return close.dropna()

def make_features(close, vol_window=20):
    """Daily log return + rolling annualised volatility."""
    df = pd.DataFrame({"close": close})
    df["ret"] = np.log(df["close"]).diff()                                   # log return
    df["vol"] = df["ret"].rolling(vol_window).std() * np.sqrt(252)           # annualised volatility
    return df.dropna()

REGIME_NAMES = ["Calm", "Turbulent", "Crisis"]
SPLIT_DATE = "2019-01-01"        # everything before = training data, from here on = test data

def fit_hmm(df, n_states=3, seed=42):
    """Fit an HMM on TRAINING data only. Returns (model, mean, std, order)."""
    X = df[["ret", "vol"]].values
    train = df.index < SPLIT_DATE
    mean, std = X[train].mean(axis=0), X[train].std(axis=0)
    Xs = (X - mean) / std
    model = GaussianHMM(n_components=n_states, covariance_type="full",
                        n_iter=500, random_state=seed)
    model.fit(Xs[train])
    order = np.argsort(model.means_[:, 1])        # sort states by volatility: low -> high
    return model, mean, std, order

def hmm_labels(df, model, mean, std, order):
    """Most likely regime for every day (Viterbi), renamed so 0=Calm, 1=Turbulent, 2=Crisis."""
    Xs = (df[["ret", "vol"]].values - mean) / std
    raw = model.predict(Xs)
    rename = {old: new for new, old in enumerate(order)}
    return np.array([rename[s] for s in raw])

nifty = load_close("^NSEI")
df = make_features(nifty)
hmm, mean, std, order = fit_hmm(df)
df["regime"] = hmm_labels(df, hmm, mean, std, order)
print(df["regime"].value_counts().rename(index=dict(enumerate(REGIME_NAMES))))

## 2. Turn the time series into training examples
Each example is a **window**: the last 20 days of [return, volatility] (20 × 2 = 40 numbers, flattened into one vector).
The label is the regime on the **next** day. Sliding the window one day at a time gives thousands of examples.

We split **by time**, never randomly (a random split would leak future information into training):
- **train:** before 2016 · **validation:** 2016–2018 (to choose the best epoch) · **test:** 2019 onwards

In [ ]:
LOOKBACK = 20      # the network sees the last 20 days of features

def make_windows(df, mean, std):
    """X[i] = last LOOKBACK days of standardised [ret, vol] up to day t, y[i] = regime on day t+1."""
    feats = (df[["ret", "vol"]].values - mean) / std
    regime = df["regime"].values
    X, y, today, dates = [], [], [], []
    for t in range(LOOKBACK - 1, len(df) - 1):
        X.append(feats[t - LOOKBACK + 1 : t + 1].flatten())
        y.append(regime[t + 1])       # target: TOMORROW's regime
        today.append(regime[t])       # today's regime (used for the simple baseline)
        dates.append(df.index[t + 1])
    return (np.array(X, dtype=np.float32), np.array(y), np.array(today), pd.DatetimeIndex(dates))

class RegimeMLP(nn.Module):
    def __init__(self, n_inputs, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_inputs, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, n_classes),          # raw scores ("logits") for each regime
        )
    def forward(self, x):
        return self.net(x)

X, y, y_today, dates = make_windows(df, mean, std)
is_train = dates < "2016-01-01"
is_val = (dates >= "2016-01-01") & (dates < SPLIT_DATE)
is_test = dates >= SPLIT_DATE

X_train, y_train = X[is_train], y[is_train]
X_val, y_val = X[is_val], y[is_val]
X_test, y_test = X[is_test], y[is_test]
print("train", X_train.shape, " val", X_val.shape, " test", X_test.shape)

## 3. The model and the training loop
A small **multi-layer perceptron (MLP)**: 40 inputs → 64 → 32 → 3 outputs (one score per regime).
The training loop is the standard four steps you saw in the PyTorch tutorial:
zero gradients → forward pass → `loss.backward()` → `optimizer.step()`.

In [ ]:
def train_model(X_train, y_train, X_val, y_val, epochs=60, lr=1e-3, verbose=True):
    model = RegimeMLP(X_train.shape[1])
    # Crisis days are rare, so we weight each class by 1 / (how often it appears).
    counts = np.bincount(y_train, minlength=3)
    weights = torch.tensor(len(y_train) / (3 * np.maximum(counts, 1)), dtype=torch.float32)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
                        batch_size=64, shuffle=True)
    Xv, yv = torch.tensor(X_val), torch.tensor(y_val)
    history, best_val, best_state = [], float("inf"), None

    for epoch in range(epochs):
        model.train()
        total = 0.0
        for xb, yb in loader:
            opt.zero_grad()                     # 1. clear old gradients
            loss = loss_fn(model(xb), yb)       # 2. forward pass + loss
            loss.backward()                     # 3. backpropagation
            opt.step()                          # 4. update the weights
            total += loss.item() * len(xb)
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xv), yv).item()
        history.append((total / len(X_train), val_loss))
        if val_loss < best_val:                 # keep the best model on validation data
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if verbose and (epoch + 1) % 10 == 0:
            print(f"epoch {epoch+1:3d}  train loss {history[-1][0]:.3f}  val loss {val_loss:.3f}")

    model.load_state_dict(best_state)
    return model, history

model, history = train_model(X_train, y_train, X_val, y_val)

hist = np.array(history)
plt.plot(hist[:, 0], label="train loss"); plt.plot(hist[:, 1], label="validation loss")
plt.xlabel("epoch"); plt.legend(); plt.title("Training curves"); plt.show()

**Reading the curves:** if training loss keeps falling while validation loss rises, the model is **overfitting**
(memorising the training data). That is why we keep the epoch with the lowest validation loss.

## 4. Test results vs the simple baseline

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(torch.tensor(X_test)).argmax(dim=1).numpy()

baseline = y_today[is_test]          # "tomorrow = today"
print(f"Neural network accuracy:            {accuracy_score(y_test, y_pred):.1%}")
print(f"Baseline ('same as today') accuracy: {accuracy_score(y_test, baseline):.1%}")

print("\nConfusion matrix (rows = true regime, columns = predicted):")
print(pd.DataFrame(confusion_matrix(y_test, y_pred, labels=[0, 1, 2]),
                   index=REGIME_NAMES, columns=REGIME_NAMES))
print()
print(classification_report(y_test, y_pred, labels=[0, 1, 2], target_names=REGIME_NAMES, zero_division=0))

In [ ]:
test_dates = dates[is_test]
plt.figure(figsize=(12, 3.5))
plt.step(test_dates, y_test, where="post", label="HMM regime (teacher)", lw=2)
plt.step(test_dates, y_pred, where="post", label="NN prediction (student)", alpha=0.7)
plt.yticks([0, 1, 2], REGIME_NAMES); plt.legend(); plt.title("Test period: true vs predicted regime")
plt.show()

## 5. How to interpret this
- Expect the baseline to be **very** strong, because regimes rarely change from one day to the next.
  Overall accuracy is therefore a weak test. The interesting question is whether the network predicts
  **regime changes**, especially the move into Crisis, earlier than the baseline.
- Look at the **Crisis row** of the confusion matrix and the recall for Crisis. Missing a crisis is far more
  costly than a false alarm.
- Whatever the result, it is a valid finding. "A simple rule is hard to beat" is a real lesson in quant finance.

Notebook 04 checks the question that actually matters: **does using these predictions improve a portfolio?**

## Check your understanding
1. Why do we split the data by time instead of randomly?
2. What are the input and the output of the network? What shapes are `X_train` and `y_train`?
3. Why do we weight the classes in the loss function?
4. Why is "tomorrow = today" such a strong baseline here?
5. What could you add to help the network spot regime changes earlier? (More features? A longer window? An LSTM?)